# End-to-End Platform Architecture & Evaluation

## Objective

This notebook consolidates the complete Retail Growth Decision Platform.

Rather than introducing another technology, I use this final notebook to evaluate how the individual components work together across the data, modeling, decisioning, serving, reliability, and governance layers.

The project began with historical retail transaction data and an uplift-modeling problem. It evolved into an end-to-end system containing a Snowflake/dbt warehouse, machine-learning workflow, economic decision engine, model tracking, API serving layer, orchestration, incremental ingestion, continuous integration, streaming processing, drift monitoring, and governance controls.

The objective of this notebook is to:

1. document the final architecture,
2. summarize verified system results,
3. connect each technology to the problem it solves,
4. identify architectural tradeoffs and limitations, and
5. verify that the major platform components remain present and internally consistent.

In [ ]:
# ============================================================
# Run the final repository/platform audit
#
# This provides a concise structural check of the components
# built throughout the project.
# ============================================================

from pathlib import Path
import subprocess
import sys


PROJECT_ROOT = next(
    path
    for path in [
        Path.cwd(),
        *Path.cwd().parents,
    ]
    if (
        path
        / "dbt"
        / "dbt_project.yml"
    ).exists()
)


result = subprocess.run(
    [
        sys.executable,
        "scripts/final_project_audit.py",
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)


print(
    result.stdout
)


if result.returncode != 0:

    print(
        result.stderr
    )

    raise RuntimeError(
        "Final project audit failed."
    )

## 1. Platform results

### Data foundation

- Raw customers: **400,162**
- Raw products: **43,038**
- Raw purchase-item rows: **45,786,568**
- Valid customer-transactions: **8,045,229**
- Purchasing customers: **400,162**
- Historical activity window: **118 days**
- Customer feature mart: **400,162 rows**
- Uplift training population: **200,039**
- Uplift scoring population: **200,123**

### Uplift-model development

Selected decisioning model: **Logistic-regression T-learner**

Development holdout:

- Treatment ROC-AUC: **0.765062**
- Control ROC-AUC: **0.772814**
- Treatment Brier score: **0.186098**
- Control Brier score: **0.188313**
- Qini: **168.366182**
- Observed uplift in top 30%: **0.063582**

The boosted T-learner produced stronger ordinary outcome-prediction ROC-AUC and Brier scores, but the logistic T-learner produced the stronger Qini and top-30% uplift metrics used for the project's decisioning objective.

### Economic decisioning

Main scenario:

- Conversion value: **20**
- Contact cost: **0.25**
- Budget: **10,000**
- Customers selected: **40,000**
- Modeled incremental conversions: **3,291.91**
- Modeled incremental value: **65,838.15**
- Contact spend: **10,000**
- Modeled net value: **55,838.15**

These values are model-based estimates under illustrative economic assumptions rather than realized campaign results.

### Warehouse reliability

The orchestrated dbt build completed successfully with:

- DAG tasks: **4**
- Total dbt resources: **64**
- Successful model/resource executions: **17**
- Passing dbt tests: **47**
- Required modeling marts verified: **3**

### Incremental ingestion

The isolated synthetic batch-ingestion demonstration implemented:

- schema validation
- raw preservation
- source provenance
- duplicate protection
- quarantine handling
- idempotent trusted-event promotion

Synthetic records remained isolated from the historical X5 warehouse and model-training data.

### Continuous integration

GitHub Actions validates:

- repository hygiene
- Python syntax
- Python unit tests
- dbt static parsing
- Docker image construction

The workflow runs on a clean hosted runner without requiring Snowflake or AWS credentials.

### Streaming

The Kafka/Spark demonstration processed synthetic events using:

- Kafka topics and partitions
- Kafka offsets
- Spark Structured Streaming
- JSON validation
- quarantine handling
- event-time watermarking
- stateful event-ID deduplication
- streaming checkpoints

Initial demonstration:

- Kafka messages: **7**
- Trusted unique events: **4**
- Quarantined events: **2**
- Duplicate suppressed: **1**

### Drift monitoring

Reference population:

**200,039 customers**

Current scoring population:

**200,123 customers**

Model features monitored:

**34**

Observed benchmark comparison:

- OK features: **34**
- WARNING features: **0**
- CRITICAL features: **0**
- Highest feature PSI: **0.000204**
- Prediction PSI: **0.000086**
- Prediction drift status: **OK**

Controlled synthetic shift:

- CRITICAL features: **2**
- WARNING features: **1**
- Synthetic prediction PSI: **0.17661**
- Synthetic prediction status: **WARNING**

### Governance

- dbt lineage targets: **3**
- Lineage resources recovered: **40**
- Model feature contract: **34 features**
- Model artifact SHA-256 fingerprint: **generated**
- Model card: **created**
- Successful Snowflake queries inspected: **150**
- Query-history window: **6 days**
- Total query execution time: **164.00 seconds**
- Average successful query duration: **1.09 seconds**
- Slowest successful query: **26.22 seconds**
- Total data scanned: **12.387 GB**
- Largest individual scan: **2.309 GB**

## 2. Technology-to-problem mapping

| Technology | Problem it solves |
|---|---|
| Amazon S3 | Durable storage for source files |
| Snowflake | Scalable analytical warehouse |
| dbt | Version-controlled SQL transformations and testing |
| Python / pandas | Analysis and ML workflow development |
| scikit-learn | Uplift-model implementation |
| MLflow | Experiment and model provenance |
| FastAPI | Programmatic model access |
| Docker | Reproducible serving environment |
| Airflow | Data-pipeline orchestration |
| GitHub Actions | Automated software-quality validation |
| Kafka | Durable event transport |
| Spark Structured Streaming | Stateful processing of event streams |
| PSI / SMD monitoring | Feature and prediction stability checks |
| dbt manifest | Machine-readable data lineage |
| SHA-256 | Exact model-artifact fingerprint |